In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [6]:
import requests
import json
import time
import pandas as pd
from datetime import datetime
import re
import logging
import os
from typing import List, Dict, Optional, Tuple
from urllib.parse import urlparse
from collections import defaultdict
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import threading
from dotenv import load_dotenv

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class StablePeriodFoodScraper:
    def __init__(self):
        # .env 파일에서 API 키 로드
        load_dotenv()
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        if not self.client_id or not self.client_secret:
            raise ValueError("API 키를 .env 파일에서 찾을 수 없습니다. Client_ID와 Client_Secret을 확인해주세요.")
        
        self.base_url = "https://openapi.naver.com/v1/search"
        self.headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret,
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        # 안정적 수집을 위한 제한값
        self.max_safe_api_calls = 15000  # 안전한 API 호출 한도
        self.target_total_images = 8000  # 목표 총 수집량
        self.api_call_count = 0
        self.collected_count = 0
        
        # 수집 상태
        self.is_collecting = False
        self.should_stop = False
        
        # 음식 키워드 (CSV에서 로드된 순서 유지)
        self.food_keywords = []
        self.csv_file_path = None

    def load_food_keywords_from_csv(self, csv_file_path: str) -> List[str]:
        """CSV 파일에서 상세메뉴를 추출하여 음식 키워드 리스트 생성 (시스템 순서 유지)"""
        try:
            df = pd.read_csv(csv_file_path, encoding='utf-8')
            logger.info(f"CSV 파일 로드: {len(df)}개 행")
            
            # 상세메뉴 컬럼에서 키워드 추출 (원본 순서 유지)
            detail_menus = df['상세메뉴'].dropna().tolist()
            
            # 시스템 제공 순서 유지하면서 키워드 추출
            ordered_keywords = []
            seen = set()
            
            for menu_group in detail_menus:
                if isinstance(menu_group, str):
                    # 쉼표로 분리 (순서 유지)
                    individual_menus = [menu.strip() for menu in menu_group.split(',')]
                    
                    for menu in individual_menus:
                        if menu and len(menu) > 1:
                            # 특수문자 제거
                            clean_keyword = re.sub(r'[^\w가-힣\s]', '', menu).strip()
                            
                            # 중복이 아니면서 유효한 키워드인 경우 순서대로 추가
                            if clean_keyword and len(clean_keyword) > 1 and clean_keyword not in seen:
                                ordered_keywords.append(clean_keyword)
                                seen.add(clean_keyword)
            
            logger.info(f"추출된 키워드: {len(ordered_keywords)}개 (시스템 순서 유지)")
            logger.info(f"상위 10개: {ordered_keywords[:10]}")
            return ordered_keywords
            
        except Exception as e:
            logger.error(f"CSV 파일 로드 실패: {e}")
            # 기본 키워드 반환 (시스템 순서)
            return ['제육볶음', '된장찌개', '콩나물무침', '계란말이', '미역국', 
                   '부대찌개', '감자탕', '순두부찌개', '설렁탕', '육개장']

    def get_keyword_search_volume(self, keyword: str, target_year: str = "2024", target_month: str = "06") -> int:
        """특정 시점의 키워드별 검색량 조사 (기록용)"""
        try:
            url = f"{self.base_url}/webkr"
            # 특정 시점 검색량 조사를 위한 쿼리
            params = {
                'query': f"{keyword} 음식 {target_year}년 {int(target_month)}월",
                'display': 1
            }
            
            response = requests.get(url, headers=self.headers, params=params, timeout=10)
            self.api_call_count += 1
            
            if response.status_code == 200:
                result = response.json()
                total_count = result.get('total', 0)
                logger.debug(f"{keyword} ({target_year}-{target_month}) 검색량: {total_count:,}")
                return total_count
            else:
                logger.warning(f"{keyword} 검색량 조사 실패: {response.status_code}")
                return 0
                
        except Exception as e:
            logger.debug(f"{keyword} 검색량 조사 오류: {e}")
            return 0

    def collect_search_volumes_for_record(self, keywords: List[str], target_year: str, target_month: str, 
                                        progress_callback=None) -> Dict[str, int]:
        """검색량 조사 (기록용, 수집 순서에는 영향 없음)"""
        logger.info(f"검색량 조사 시작 ({target_year}-{target_month} 기준)...")
        keyword_volumes = {}
        
        for i, keyword in enumerate(keywords, 1):
            if self.should_stop:
                break
                
            volume = self.get_keyword_search_volume(keyword, target_year, target_month)
            keyword_volumes[keyword] = volume
            
            # 진행률 콜백 (검색량 조사 단계)
            if progress_callback:
                search_progress = (i / len(keywords)) * 20  # 전체 진행률의 20%를 검색량 조사에 할당
                progress_callback(search_progress, 0, self.api_call_count, f"검색량 조사: {keyword}")
            
            logger.info(f"[{i:3d}/{len(keywords)}] {keyword}: {volume:,}")
            time.sleep(0.3)  # API 호출 간격
        
        # 검색량 통계 출력 (참고용)
        sorted_by_volume = sorted(keyword_volumes.items(), key=lambda x: x[1], reverse=True)
        logger.info(f"\n검색량 상위 15개 키워드 ({target_year}-{target_month} 기준):")
        for i, (keyword, volume) in enumerate(sorted_by_volume[:15], 1):
            logger.info(f"  {i:2d}. {keyword}: {volume:,}")
        
        logger.info("검색량 조사 완료. 시스템 순서대로 수집 시작...")
        return keyword_volumes

    def generate_period_search_queries(self, target_year: str, target_month: str) -> List[str]:
        """특정 년월에 맞는 검색 쿼리 생성"""
        month_int = int(target_month)
        year_short = target_year[2:]  # 24, 25
        
        # 기본 날짜 쿼리
        date_queries = [
            f"{target_year}년 {month_int}월",
            f"{target_year}.{target_month}",
            f"{year_short}년 {month_int}월",
            f"{target_year}/{target_month}",
            f"{target_year}{target_month}",
            f"{month_int}월 {target_year}"
        ]
        
        # 계절별 키워드 추가
        seasonal_keywords = {
            '01': ['신년', '1월', '겨울'],
            '02': ['2월', '겨울', '설날'],
            '03': ['3월', '봄', '초봄'],
            '04': ['4월', '봄', '중봄'],
            '05': ['5월', '봄', '늦봄'],
            '06': ['6월', '여름', '초여름'],
            '07': ['7월', '여름', '중여름'],
            '08': ['8월', '여름', '늦여름'],
            '09': ['9월', '가을', '초가을'],
            '10': ['10월', '가을', '중가을'],
            '11': ['11월', '가을', '늦가을'],
            '12': ['12월', '겨울', '연말']
        }
        
        if target_month in seasonal_keywords:
            date_queries.extend(seasonal_keywords[target_month])
        
        return date_queries

    def is_relevant_to_keyword(self, item: Dict, keyword: str) -> bool:
        """검색 결과가 해당 키워드와 관련이 있는지 검증"""
        title = item.get('title', '').replace('<b>', '').replace('</b>', '').lower()
        url = item.get('link', '').lower()
        description = item.get('description', '').lower()
        
        keyword_lower = keyword.lower()
        
        # 키워드가 제목, URL, 설명에 포함되어 있는지 확인
        if keyword_lower in title or keyword_lower in url or keyword_lower in description:
            return True
        
        # 음식 관련 단어가 있는지 추가 확인
        food_indicators = ['음식', '요리', '레시피', '맛', '식당', '메뉴']
        if any(indicator in title for indicator in food_indicators):
            return True
        
        return False

    def search_keyword_thoroughly(self, keyword: str, target_year: str, target_month: str) -> List[Dict]:
        """키워드별 철저한 검색 (제한 없이 가능한 많이 수집)"""
        if self.should_stop:
            return []
            
        results = []
        seen_urls = set()
        period_queries = self.generate_period_search_queries(target_year, target_month)
        
        logger.info(f"수집 중: {keyword} ({target_year}-{target_month})")
        
        # 검색 방법별로 수집
        search_methods = [
            ('image', 'sim'),   # 이미지 검색 - 유사도순
            ('image', 'date'),  # 이미지 검색 - 날짜순
            ('blog', 'date'),   # 블로그 검색 - 날짜순
            ('blog', 'sim'),    # 블로그 검색 - 유사도순
        ]
        
        keyword_collected = 0
        
        for search_type, sort_type in search_methods:
            if self.should_stop:
                break
                
            if self.api_call_count >= self.max_safe_api_calls:
                logger.warning("안전한 API 호출 한도 도달")
                self.should_stop = True
                break
            
            if self.collected_count >= self.target_total_images:
                logger.info("목표 총 수집량 도달")
                self.should_stop = True
                break
            
            # 각 검색 방법에 대해 모든 기간 쿼리 시도
            for period_query in period_queries:
                if self.should_stop or self.collected_count >= self.target_total_images:
                    break
                
                query_results = self.search_with_specific_method(
                    keyword, period_query, search_type, sort_type, seen_urls
                )
                
                # 관련성 검증 후 추가
                for result in query_results:
                    if self.collected_count >= self.target_total_images:
                        self.should_stop = True
                        break
                        
                    if self.is_relevant_to_keyword(result, keyword):
                        results.append(result)
                        keyword_collected += 1
                        self.collected_count += 1
                
                time.sleep(0.2)  # 안정적인 대기 시간
        
        logger.info(f"{keyword}: {keyword_collected}개 수집 완료 (총 수집량: {self.collected_count})")
        return results

    def search_with_specific_method(self, keyword: str, period_query: str, 
                                  search_type: str, sort_type: str, seen_urls: set) -> List[Dict]:
        """특정 방법으로 검색 수행"""
        results = []
        
        if search_type == 'image':
            url = f"{self.base_url}/image"
            full_query = f"{keyword} 음식 이미지 {period_query}"
        else:  # blog
            url = f"{self.base_url}/blog"
            full_query = f"{keyword} 음식 {period_query}"
        
        # API 제한 내에서 최대한 검색 (1~1000)
        for start_pos in range(1, 1001, 100):
            if self.should_stop or self.api_call_count >= self.max_safe_api_calls:
                break
                
            if self.collected_count >= self.target_total_images:
                break
                
            params = {
                'query': full_query,
                'display': 100,
                'start': start_pos,
                'sort': sort_type
            }
            
            try:
                response = requests.get(url, headers=self.headers, params=params, timeout=10)
                self.api_call_count += 1
                
                if response.status_code == 200:
                    result = response.json()
                    items = result.get('items', [])
                    
                    if not items:  # 더 이상 결과가 없으면 중단
                        break
                    
                    for item in items:
                        item_url = item.get('link', '')
                        
                        if item_url and item_url not in seen_urls:
                            seen_urls.add(item_url)
                            
                            # 결과 객체 생성
                            result_obj = {
                                'keyword': keyword,
                                'title': item.get('title', '').replace('<b>', '').replace('</b>', '').strip(),
                                'image_url': item_url,
                                'thumbnail_url': item.get('thumbnail', ''),
                                'description': item.get('description', '').replace('<b>', '').replace('</b>', '').strip(),
                                'search_method': f"{search_type}_{sort_type}",
                                'period_query': period_query,
                                'domain': self.extract_domain(item_url),
                                'collected_at': datetime.now().isoformat(),
                                'api_call_number': self.api_call_count,
                                'collection_order': len(results) + 1
                            }
                            results.append(result_obj)
                
                elif response.status_code == 429:
                    logger.warning("API 요청 한도 초과, 대기 중...")
                    time.sleep(3)
                    
            except Exception as e:
                logger.debug(f"검색 오류: {e}")
                continue
            
            time.sleep(0.1)  # 안전한 대기
        
        return results

    def extract_domain(self, url: str) -> str:
        """URL에서 도메인 추출"""
        try:
            return urlparse(url).netloc.lower()
        except:
            return "unknown"

    def collect_images_by_system_order_with_search_volume_record(self, target_year: str, target_month: str, 
                                                              progress_callback=None) -> pd.DataFrame:
        """시스템 순서대로 수집하되 검색량은 기록"""
        if not self.food_keywords:
            logger.error("음식 키워드가 로드되지 않았습니다")
            return pd.DataFrame()
        
        self.is_collecting = True
        self.should_stop = False
        self.api_call_count = 0
        self.collected_count = 0
        
        all_results = []
        
        logger.info(f"시스템 순서 기반 수집 시작: {target_year}-{target_month}")
        logger.info(f"목표 총 수집량: {self.target_total_images:,}개")
        logger.info(f"키워드 수: {len(self.food_keywords)}개 (시스템 순서 유지)")
        
        # 1. 검색량 조사 (기록용, 수집 순서에 영향 없음)
        keyword_volumes = self.collect_search_volumes_for_record(
            self.food_keywords, target_year, target_month, progress_callback
        )
        
        if not keyword_volumes:
            logger.error("검색량 조사 실패")
            self.is_collecting = False
            return pd.DataFrame()
        
        total_keywords = len(self.food_keywords)
        logger.info(f"키워드 순서: {self.food_keywords[:10]}... (시스템 제공 순서)")
        
        # 2. 시스템 제공 순서대로 키워드 처리
        for i, keyword in enumerate(self.food_keywords, 1):
            if self.should_stop:
                logger.info("수집 중단됨")
                break
                
            if self.collected_count >= self.target_total_images:
                logger.info(f"목표 수집량 달성! (총 {self.collected_count:,}개)")
                break
                
            if self.api_call_count >= self.max_safe_api_calls:
                logger.info(f"안전한 API 호출량 도달! (총 {self.api_call_count:,}회)")
                break
            
            # 해당 키워드의 검색량 정보 가져오기
            search_volume = keyword_volumes.get(keyword, 0)
            
            # 키워드별 철저한 수집
            logger.info(f"[{i:3d}/{total_keywords}] 수집 중: {keyword} (검색량: {search_volume:,})")
            keyword_results = self.search_keyword_thoroughly(keyword, target_year, target_month)
            
            # 검색량 정보를 각 결과에 추가
            for result in keyword_results:
                result['keyword_search_volume'] = search_volume
                result['keyword_system_order'] = i
            
            all_results.extend(keyword_results)
            
            # 진행률 콜백 (검색량 조사 20% + 수집 80%)
            if progress_callback:
                collection_progress = (self.collected_count / self.target_total_images) * 80
                keyword_progress = (i / total_keywords) * 80
                total_progress = 20 + min(max(collection_progress, keyword_progress), 80)
                progress_callback(total_progress, self.collected_count, self.api_call_count, 
                                f"수집 중: {keyword}")
            
            # 진행률 로그
            if i % 5 == 0 or self.collected_count >= self.target_total_images:
                remaining = self.target_total_images - self.collected_count
                logger.info(f"진행률: {i}/{total_keywords} 키워드 처리 "
                           f"| 수집량: {self.collected_count:,}/{self.target_total_images:,} "
                           f"| 남은 수집량: {remaining:,} "
                           f"| API: {self.api_call_count:,}회")
            
            # 목표 달성 체크
            if self.collected_count >= self.target_total_images:
                logger.info(f"목표 수집량 달성! 키워드 '{keyword}' (시스템 순서: {i}번째)에서 완료")
                break
            
            # 안전한 대기 (키워드 간)
            time.sleep(0.3)
        
        self.is_collecting = False
        
        # DataFrame 생성 및 정리
        df = pd.DataFrame(all_results)
        if not df.empty:
            # 중복 제거 (URL 기준, 먼저 수집된 것 유지)
            original_count = len(df)
            df = df.drop_duplicates(subset=['image_url'], keep='first')
            duplicate_count = original_count - len(df)
            
            # 시스템 순서대로 정렬 (수집 순서 유지)
            df = df.sort_values(['keyword_system_order', 'collected_at']).reset_index(drop=True)
            
            logger.info(f"중복 제거: {duplicate_count:,}개")
            logger.info(f"최종 수집량: {len(df):,}개")
        
        return df

    def stop_collection(self):
        """수집 중단"""
        self.should_stop = True
        logger.info("수집 중단 요청됨")

class FoodScraperGUI:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("음식 이미지 수집기 (시스템 순서 + 검색량 기록)")
        self.root.geometry("650x580")
        
        self.scraper = None
        self.collection_thread = None
        self.setup_ui()
        
    def setup_ui(self):
        """UI 구성"""
        # API 키 상태 표시 프레임
        api_status_frame = ttk.LabelFrame(self.root, text="API 키 상태", padding=10)
        api_status_frame.pack(fill="x", padx=10, pady=5)
        
        self.api_status_var = tk.StringVar()
        self.check_api_keys()
        ttk.Label(api_status_frame, textvariable=self.api_status_var).pack()
        
        # CSV 파일 선택 프레임
        csv_frame = ttk.LabelFrame(self.root, text="음식 키워드 파일", padding=10)
        csv_frame.pack(fill="x", padx=10, pady=5)
        
        self.csv_file_var = tk.StringVar()
        ttk.Entry(csv_frame, textvariable=self.csv_file_var, width=60).grid(row=0, column=0)
        ttk.Button(csv_frame, text="파일 선택", command=self.select_csv_file).grid(row=0, column=1, padx=(10,0))
        
        # 수집 기간 설정 프레임
        period_frame = ttk.LabelFrame(self.root, text="수집 기간 설정", padding=10)
        period_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(period_frame, text="년도:").grid(row=0, column=0, sticky="w")
        self.year_var = tk.StringVar(value="2024")
        year_combo = ttk.Combobox(period_frame, textvariable=self.year_var, 
                                 values=["2024", "2025", "2026"], width=10)
        year_combo.grid(row=0, column=1, padx=(10,0))
        
        ttk.Label(period_frame, text="월:").grid(row=0, column=2, sticky="w", padx=(20,0))
        self.month_var = tk.StringVar(value="06")
        month_combo = ttk.Combobox(period_frame, textvariable=self.month_var, 
                                  values=[f"{i:02d}" for i in range(1, 13)], width=10)
        month_combo.grid(row=0, column=3, padx=(10,0))
        
        # 수집 설정 프레임
        settings_frame = ttk.LabelFrame(self.root, text="수집 설정", padding=10)
        settings_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(settings_frame, text="목표 수집량:").grid(row=0, column=0, sticky="w")
        self.target_amount_var = tk.StringVar(value="8000")
        ttk.Entry(settings_frame, textvariable=self.target_amount_var, width=10).grid(row=0, column=1, padx=(10,0))
        ttk.Label(settings_frame, text="개").grid(row=0, column=2, sticky="w")
        
        ttk.Label(settings_frame, text="수집 방식: 시스템 순서대로 수집 (검색량은 기록용)").grid(row=1, column=0, columnspan=3, sticky="w", pady=(10,0))
        
        # 수집 제어 프레임
        control_frame = ttk.LabelFrame(self.root, text="수집 제어", padding=10)
        control_frame.pack(fill="x", padx=10, pady=5)
        
        self.start_button = ttk.Button(control_frame, text="수집 시작", command=self.start_collection)
        self.start_button.pack(side="left", padx=(0,10))
        
        self.stop_button = ttk.Button(control_frame, text="수집 중단", command=self.stop_collection, state="disabled")
        self.stop_button.pack(side="left")
        
        # 진행률 표시 프레임
        progress_frame = ttk.LabelFrame(self.root, text="진행 상황", padding=10)
        progress_frame.pack(fill="x", padx=10, pady=5)
        
        self.progress_var = tk.DoubleVar()
        self.progress_bar = ttk.Progressbar(progress_frame, variable=self.progress_var, maximum=100)
        self.progress_bar.pack(fill="x", pady=(0,10))
        
        self.status_var = tk.StringVar(value="대기 중...")
        ttk.Label(progress_frame, textvariable=self.status_var).pack()
        
        # 결과 표시 프레임
        result_frame = ttk.LabelFrame(self.root, text="수집 결과", padding=10)
        result_frame.pack(fill="both", expand=True, padx=10, pady=5)
        
        self.result_text = tk.Text(result_frame, height=12, wrap="word")
        scrollbar = ttk.Scrollbar(result_frame, orient="vertical", command=self.result_text.yview)
        self.result_text.configure(yscrollcommand=scrollbar.set)
        
        self.result_text.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
    def check_api_keys(self):
        """API 키 상태 확인"""
        try:
            load_dotenv()
            client_id = os.getenv('Client_ID')
            client_secret = os.getenv('Client_Secret')
            
            if client_id and client_secret:
                self.api_status_var.set(f"✓ API 키 로드됨 (ID: {client_id[:10]}...)")
            else:
                self.api_status_var.set("✗ .env 파일에서 API 키를 찾을 수 없습니다")
        except Exception as e:
            self.api_status_var.set(f"✗ API 키 로드 오류: {e}")
            
    def select_csv_file(self):
        """CSV 파일 선택"""
        file_path = filedialog.askopenfilename(
            title="음식 키워드 CSV 파일 선택",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )
        if file_path:
            self.csv_file_var.set(file_path)
            
    def start_collection(self):
        """수집 시작"""
        if not self.validate_inputs():
            return
            
        # UI 상태 변경
        self.start_button.config(state="disabled")
        self.stop_button.config(state="normal")
        self.result_text.delete(1.0, tk.END)
        
        # 스크래퍼 초기화
        try:
            self.scraper = StablePeriodFoodScraper()
            
            # 목표 수집량 설정
            try:
                target_amount = int(self.target_amount_var.get())
                self.scraper.target_total_images = target_amount
            except ValueError:
                messagebox.showerror("오류", "목표 수집량은 숫자여야 합니다")
                self.reset_ui()
                return
            
        except ValueError as e:
            messagebox.showerror("API 키 오류", str(e))
            self.reset_ui()
            return
        
        # CSV 파일 로드
        csv_file = self.csv_file_var.get().strip()
        if csv_file:
            self.scraper.food_keywords = self.scraper.load_food_keywords_from_csv(csv_file)
            self.log_message(f"키워드 로드: {len(self.scraper.food_keywords)}개")
        else:
            messagebox.showerror("오류", "CSV 파일을 선택해주세요")
            self.reset_ui()
            return
            
        # 수집 스레드 시작
        self.collection_thread = threading.Thread(
            target=self.run_collection,
            args=(self.year_var.get(), self.month_var.get())
        )
        self.collection_thread.daemon = True
        self.collection_thread.start()
        
    def validate_inputs(self):
        """입력값 검증"""
        # .env에서 API 키 확인
        load_dotenv()
        client_id = os.getenv('Client_ID')
        client_secret = os.getenv('Client_Secret')
        
        if not client_id or not client_secret:
            messagebox.showerror("오류", ".env 파일에 Client_ID와 Client_Secret을 설정해주세요")
            return False
            
        return True
        
    def run_collection(self, year: str, month: str):
        """수집 실행"""
        try:
            self.log_message(f"수집 시작: {year}-{month}")
            self.log_message(f"목표 수집량: {self.scraper.target_total_images:,}개")
            self.log_message(f"수집 방식: 시스템 순서대로 수집 (검색량은 기록용)")
            
            result_df = self.scraper.collect_images_by_system_order_with_search_volume_record(
                year, month, self.update_progress
            )
            
            if not result_df.empty:
                # 결과 저장
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                filename = f"food_images_system_order_with_volume_{year}{month}_{timestamp}.csv"
                result_df.to_csv(filename, index=False, encoding='utf-8-sig')
                
                # 통계 출력
                self.log_message(f"\n수집 완료!")
                self.log_message(f"파일 저장: {filename}")
                self.log_message(f"총 이미지: {len(result_df):,}개")
                self.log_message(f"고유 키워드: {result_df['keyword'].nunique()}개")
                self.log_message(f"API 호출: {self.scraper.api_call_count:,}회")
                self.log_message(f"수집 효율: {len(result_df)/self.scraper.api_call_count:.2f}개/호출")
                
                # 키워드별 수집량 (시스템 순서)
                self.log_message(f"\n키워드별 수집량 (시스템 순서):")
                if 'keyword_system_order' in result_df.columns and 'keyword_search_volume' in result_df.columns:
                    keyword_stats = result_df.groupby(['keyword', 'keyword_system_order', 'keyword_search_volume']).size().reset_index(name='count')
                    keyword_stats = keyword_stats.sort_values('keyword_system_order')
                    
                    for i, row in keyword_stats.head(10).iterrows():
                        self.log_message(f"  {row['keyword_system_order']:2d}. {row['keyword']}: {row['count']}개 (검색량: {row['keyword_search_volume']:,})")
                else:
                    keyword_counts = result_df['keyword'].value_counts()
                    for i, (keyword, count) in enumerate(keyword_counts.head(10).items(), 1):
                        self.log_message(f"  {i:2d}. {keyword}: {count}개")
                
                # 검색 방법별 분포
                method_counts = result_df['search_method'].value_counts()
                self.log_message(f"\n검색 방법별 분포:")
                for method, count in method_counts.items():
                    self.log_message(f"  {method}: {count}개")
                    
            else:
                self.log_message("수집된 데이터가 없습니다")
                
        except Exception as e:
            self.log_message(f"오류 발생: {e}")
            logger.error(f"수집 중 오류: {e}")
            
        finally:
            self.root.after(0, self.reset_ui)
            
    def update_progress(self, percent: float, collected: int, api_calls: int, status_msg: str = ""):
        """진행률 업데이트"""
        self.root.after(0, lambda: self.progress_var.set(percent))
        
        # 목표 대비 진행률 표시
        target = self.scraper.target_total_images if self.scraper else 8000
        status_text = f"진행률: {percent:.1f}% | 수집: {collected:,}/{target:,}개 | API: {api_calls:,}회"
        if status_msg:
            status_text += f" | {status_msg}"
            
        self.root.after(0, lambda: self.status_var.set(status_text))
        
    def stop_collection(self):
        """수집 중단"""
        if self.scraper:
            self.scraper.stop_collection()
        self.log_message("수집 중단 요청됨...")
        
    def log_message(self, message: str):
        """로그 메시지 출력"""
        def update_text():
            self.result_text.insert(tk.END, message + "\n")
            self.result_text.see(tk.END)
            
        self.root.after(0, update_text)
        
    def reset_ui(self):
        """UI 상태 리셋"""
        self.start_button.config(state="normal")
        self.stop_button.config(state="disabled")
        self.progress_var.set(0)
        self.status_var.set("대기 중...")
        
    def run(self):
        """GUI 실행"""
        self.root.mainloop()

# 명령줄 실행용 함수
def run_command_line():
    """명령줄에서 실행"""
    print("=== 음식 이미지 수집기 (시스템 순서 + 검색량 기록) ===")
    
    # .env 파일 확인
    load_dotenv()
    client_id = os.getenv('Client_ID')
    client_secret = os.getenv('Client_Secret')
    
    if not client_id or not client_secret:
        print("오류: .env 파일에 Client_ID와 Client_Secret을 설정해주세요")
        print("예시 .env 파일:")
        print("Client_ID=your_client_id_here")
        print("Client_Secret=your_client_secret_here")
        return
    
    print(f"API 키 로드됨 (ID: {client_id[:10]}...)")
    
    # CSV 파일 경로
    csv_file = input("CSV 파일 경로 (기본값: 식당대12중53소132상세메뉴379분류.csv): ").strip()
    if not csv_file:
        csv_file = "식당대12중53소132상세메뉴379분류.csv"
    
    # 수집 기간
    year = input("수집 년도 (기본값: 2024): ").strip() or "2024"
    month = input("수집 월 (기본값: 06): ").strip() or "06"
    
    # 목표 수집량
    target_str = input("목표 수집량 (기본값: 8000): ").strip() or "8000"
    try:
        target_amount = int(target_str)
    except ValueError:
        target_amount = 8000
    
    # 스크래퍼 실행
    try:
        scraper = StablePeriodFoodScraper()
        scraper.target_total_images = target_amount
        scraper.food_keywords = scraper.load_food_keywords_from_csv(csv_file)
        
        print(f"\n수집 시작: {year}-{month}")
        print(f"목표 수집량: {target_amount:,}개")
        print(f"키워드 수: {len(scraper.food_keywords)}개")
        print(f"수집 방식: 시스템 순서대로 수집 (검색량은 기록용)")
        
        result_df = scraper.collect_images_by_system_order_with_search_volume_record(year, month)
        
        if not result_df.empty:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"food_images_system_order_with_volume_{year}{month}_{timestamp}.csv"
            result_df.to_csv(filename, index=False, encoding='utf-8-sig')
            
            print(f"\n수집 완료!")
            print(f"파일: {filename}")
            print(f"총 이미지: {len(result_df):,}개")
            print(f"API 호출: {scraper.api_call_count:,}회")
            print(f"수집 효율: {len(result_df)/scraper.api_call_count:.2f}개/호출")
            
            # 시스템 순서대로 상위 키워드 표시
            if 'keyword_system_order' in result_df.columns and 'keyword_search_volume' in result_df.columns:
                print(f"\n상위 키워드 (시스템 순서):")
                keyword_stats = result_df.groupby(['keyword', 'keyword_system_order', 'keyword_search_volume']).size().reset_index(name='count')
                keyword_stats = keyword_stats.sort_values('keyword_system_order')
                
                for i, row in keyword_stats.head(10).iterrows():
                    print(f"  {row['keyword_system_order']:2d}. {row['keyword']}: {row['count']}개 (검색량: {row['keyword_search_volume']:,})")
        else:
            print("수집된 데이터가 없습니다")
            
    except Exception as e:
        print(f"오류 발생: {e}")

def main():
    """메인 실행 함수"""
    import sys
    
    # .env 파일 생성 가이드
    if not os.path.exists('.env'):
        print("⚠️  .env 파일이 없습니다!")
        print("다음 내용으로 .env 파일을 생성해주세요:")
        print()
        print("Client_ID=your_naver_client_id")
        print("Client_Secret=your_naver_client_secret")
        print()
        print("네이버 API 키 발급: https://developers.naver.com")
        return
    
    if len(sys.argv) > 1 and sys.argv[1] == "--cli":
        run_command_line()
    else:
        # GUI 실행
        app = FoodScraperGUI()
        app.run()

if __name__ == "__main__":
    main()

2025-07-14 22:17:23,221 - INFO - CSV 파일 로드: 138개 행
2025-07-14 22:17:23,228 - INFO - 추출된 키워드: 377개 (시스템 순서 유지)
2025-07-14 22:17:23,228 - INFO - 상위 10개: ['제육볶음', '매운제육볶음', '두부제육볶음', '된장찌개', '김치찌개', '청국장찌개', '콩나물무침', '시금치나물', '도라지무침', '계란말이']
2025-07-14 22:17:23,262 - INFO - 시스템 순서 기반 수집 시작: 2024-06
2025-07-14 22:17:23,274 - INFO - 목표 총 수집량: 8,000개
2025-07-14 22:17:23,275 - INFO - 키워드 수: 377개 (시스템 순서 유지)
2025-07-14 22:17:23,276 - INFO - 검색량 조사 시작 (2024-06 기준)...
2025-07-14 22:17:23,622 - INFO - [  1/377] 제육볶음: 458,169
2025-07-14 22:17:24,464 - INFO - [  2/377] 매운제육볶음: 312,940
2025-07-14 22:17:25,288 - INFO - [  3/377] 두부제육볶음: 282,694
2025-07-14 22:17:26,107 - INFO - [  4/377] 된장찌개: 847,010
2025-07-14 22:17:26,757 - INFO - [  5/377] 김치찌개: 1,292,051
2025-07-14 22:17:27,395 - INFO - [  6/377] 청국장찌개: 487,058
2025-07-14 22:17:28,021 - INFO - [  7/377] 콩나물무침: 611,211
2025-07-14 22:17:28,626 - INFO - [  8/377] 시금치나물: 365,461
2025-07-14 22:17:29,285 - INFO - [  9/377] 도라지무침: 298,538
2025-07-14 22

2025-07-14 22:19:05,151 - INFO - [137/377] 해물탕: 929,955
2025-07-14 22:19:06,016 - INFO - [138/377] 완자탕: 272,753
2025-07-14 22:19:06,880 - INFO - [139/377] 탕수육: 650,245
2025-07-14 22:19:07,508 - INFO - [140/377] 새콤달콤탕수육: 267,089
2025-07-14 22:19:08,397 - INFO - [141/377] 깐풍기: 94,877
2025-07-14 22:19:09,279 - INFO - [142/377] 라조기: 9,347
2025-07-14 22:19:09,842 - INFO - [143/377] 깐쇼새우: 34,222
2025-07-14 22:19:10,485 - INFO - [144/377] 마요새우: 218,801
2025-07-14 22:19:11,120 - INFO - [145/377] 칠리새우: 278,129
2025-07-14 22:19:11,975 - INFO - [146/377] 유린기: 191,383
2025-07-14 22:19:12,555 - INFO - [147/377] 깐풍새우: 217,898
2025-07-14 22:19:13,157 - INFO - [148/377] 깐풍오징어: 195,701
2025-07-14 22:19:14,018 - INFO - [149/377] 마라탕: 905,503
2025-07-14 22:19:14,616 - INFO - [150/377] 마라샹궈: 196,650
2025-07-14 22:19:15,473 - INFO - [151/377] 훠궈: 237,831
2025-07-14 22:19:16,325 - INFO - [152/377] 탄탄면: 24,382
2025-07-14 22:19:17,222 - INFO - [153/377] 마라면: 12,788,557
2025-07-14 22:19:18,126 - INFO - [154/37

2025-07-14 22:20:52,433 - INFO - [283/377] 새콤매운국물: 39,029
2025-07-14 22:20:53,064 - INFO - [284/377] 치킨커리: 245,636
2025-07-14 22:20:53,878 - INFO - [285/377] 양고기커리: 291,522
2025-07-14 22:20:54,778 - INFO - [286/377] 달커리: 310,815
2025-07-14 22:20:55,637 - INFO - [287/377] 파라타: 5,671
2025-07-14 22:20:56,254 - INFO - [288/377] 탄두리치킨: 189,608
2025-07-14 22:20:57,081 - INFO - [289/377] 케밥: 137,489
2025-07-14 22:20:57,944 - INFO - [290/377] 구이: 2,801,215
2025-07-14 22:20:58,507 - INFO - [291/377] 나시고렝: 2,937
2025-07-14 22:20:59,415 - INFO - [292/377] 미고렝: 8,990
2025-07-14 22:21:00,288 - INFO - [293/377] 사테: 26,278
2025-07-14 22:21:01,107 - INFO - [294/377] 락사: 170,614
2025-07-14 22:21:02,082 - INFO - [295/377] 렌당: 882
2025-07-14 22:21:02,703 - INFO - [296/377] 후라이드치킨: 176,544
2025-07-14 22:21:03,337 - INFO - [297/377] 바삭치킨: 292,870
2025-07-14 22:21:03,960 - INFO - [298/377] 달콤양념: 463,253
2025-07-14 22:21:04,840 - INFO - [299/377] 매운양념: 790,516
2025-07-14 22:21:05,572 - INFO - [300/377] 간장치킨:

2025-07-14 22:24:45,049 - INFO - [ 15/377] 콩나물국: 1,336,075
2025-07-14 22:24:45,685 - INFO - [ 16/377] 부대찌개: 1,080,767
2025-07-14 22:24:46,496 - INFO - [ 17/377] 햄부대찌개: 469,650
2025-07-14 22:24:47,310 - INFO - [ 18/377] 치즈부대찌개: 481,825
2025-07-14 22:24:48,212 - INFO - [ 19/377] 감자탕: 1,446,778
2025-07-14 22:24:48,835 - INFO - [ 20/377] 뼈해장국: 381,272
2025-07-14 22:24:49,402 - INFO - [ 21/377] 등뼈찜: 80,215
2025-07-14 22:24:49,992 - INFO - [ 22/377] 순두부찌개: 479,742
2025-07-14 22:24:50,807 - INFO - [ 23/377] 해물순두부찌개: 373,087
2025-07-14 22:24:51,644 - INFO - [ 24/377] 버섯순두부찌개: 349,780
2025-07-14 22:24:52,465 - INFO - [ 25/377] 설렁탕: 444,462
2025-07-14 22:24:53,318 - INFO - [ 26/377] 곰탕: 808,614
2025-07-14 22:24:54,162 - INFO - [ 27/377] 갈비탕: 1,285,749
2025-07-14 22:24:55,019 - INFO - [ 28/377] 삼계탕: 1,295,458
2025-07-14 22:24:55,844 - INFO - [ 29/377] 육개장: 539,066
2025-07-14 22:24:56,660 - INFO - [ 30/377] 닭개장: 200,069
2025-07-14 22:24:57,276 - INFO - [ 31/377] 콩나물해장국: 302,609
2025-07-14 22:24:57

2025-07-14 22:26:26,374 - INFO - [160/377] 왕만두: 93,529
2025-07-14 22:26:27,206 - INFO - [161/377] 샤오롱바오: 8,715
2025-07-14 22:26:28,025 - INFO - [162/377] 하가우: 2,971
2025-07-14 22:26:28,855 - INFO - [163/377] 슈마이: 1,865
2025-07-14 22:26:29,666 - INFO - [164/377] 짬짜면: 78,851
2025-07-14 22:26:30,483 - INFO - [165/377] 탕짜면: 35,961
2025-07-14 22:26:31,325 - INFO - [166/377] 우짬탕: 30
2025-07-14 22:26:31,949 - INFO - [167/377] 볶음밥탕수육: 350,452
2025-07-14 22:26:32,560 - INFO - [168/377] 덮밥세트: 519,873
2025-07-14 22:26:33,376 - INFO - [169/377] 연어초밥: 353,184
2025-07-14 22:26:33,976 - INFO - [170/377] 참치초밥: 352,901
2025-07-14 22:26:34,666 - INFO - [171/377] 장어초밥: 433,669
2025-07-14 22:26:35,276 - INFO - [172/377] 참치사시미: 217,655
2025-07-14 22:26:35,875 - INFO - [173/377] 연어사시미: 213,075
2025-07-14 22:26:36,457 - INFO - [174/377] 모둠사시미: 199,857
2025-07-14 22:26:37,090 - INFO - [175/377] 연어덮밥: 337,491
2025-07-14 22:26:37,692 - INFO - [176/377] 장어덮밥: 347,693
2025-07-14 22:26:38,345 - INFO - [177/377] 김밥

2025-07-14 22:28:07,639 - INFO - [306/377] 순살치킨: 365,693
2025-07-14 22:28:08,289 - INFO - [307/377] 팝콘치킨: 253,403
2025-07-14 22:28:08,926 - INFO - [308/377] 치킨텐더: 207,399
2025-07-14 22:28:09,847 - INFO - [309/377] 다리: 8,260,278
2025-07-14 22:28:10,716 - INFO - [310/377] 가슴살: 416,640
2025-07-14 22:28:11,567 - INFO - [311/377] 콤비네이션: 138,989
2025-07-14 22:28:12,198 - INFO - [312/377] 고구마피자: 539,270
2025-07-14 22:28:12,791 - INFO - [313/377] 포테이토피자: 288,633
2025-07-14 22:28:13,431 - INFO - [314/377] 화이트피자: 602,061
2025-07-14 22:28:14,066 - INFO - [315/377] 불고기피자: 567,913
2025-07-14 22:28:14,688 - INFO - [316/377] 갈비피자: 782,791
2025-07-14 22:28:15,352 - INFO - [317/377] 한국식피자: 1,605,947
2025-07-14 22:28:15,996 - INFO - [318/377] 시카고피자: 347,437
2025-07-14 22:28:16,598 - INFO - [319/377] 딥디쉬: 56,256
2025-07-14 22:28:17,245 - INFO - [320/377] 두꺼운피자: 386,400
2025-07-14 22:28:17,887 - INFO - [321/377] 씬크러스트: 34,318
2025-07-14 22:28:18,548 - INFO - [322/377] 얇은피자: 501,398
2025-07-14 22:28:19,398